## F3: NLP Annotation

This notebook annotates the F2 VOCAB table with linguistic features from NLTK, producing a richer dictionary that downstream stages will use for filtering, normalization, and grouping. TOKEN is unchanged from F2 but re-saved to `data/F3/` so each pipeline stage's outputs are self-contained.

### Conceptual goal

F2 produced VOCAB as a frequency dictionary — every distinct word-form with its corpus count. F3 extends each entry with annotations describing the word's grammatical and morphological identity. These annotations don't alter the token stream; they enrich how it can be sliced. After F3, a single VOCAB row tells you not just *that* a term appears in Swift, but *what kind of word it is*, *what its root form is*, and *whether it carries topical signal or is a structural function word*.

The assignment doc places stopword flags, POS aggregations, stems, and lemmas explicitly under F3, so the boundary between "constructing the standard tables" (F2) and "annotating with NLP features" (F3) is drawn at this notebook.

### Annotations added to VOCAB

**`stop`** — boolean flag from NLTK's English stopword list. A simple lookup that marks function words ("the", "of", "and") for downstream filtering. Implemented via the M04 mapping pattern: build a one-column DataFrame of stopwords, map onto VOCAB.term_str, fill missing values with 0.

**`p_stem`** — Porter stem of `term_str`. Applied with `nltk.stem.porter.PorterStemmer`. Porter is rule-based and aggressive, normalizing 64% of terms in the corpus, often producing non-word tokens (`happy → happi`, `little → littl`, `studies → studi`). Useful when the goal is collapsing inflectional variants without caring about linguistic accuracy.

**`pos_max`** — the most common Penn Treebank POS tag for each term across the corpus. Computed by aggregating per-token POS tags (which already exist on TOKEN from F2) up to the term level via `groupby(['term_str','pos']).size().unstack().idxmax(axis=1)`. This makes POS a property of the dictionary entry itself, not just individual tokens. It also enables proper lemmatization in the next step.

**`wn_pos`** — WordNet POS tag derived from `pos_max`. NLTK's lemmatizer requires WordNet's four-tag system (`'n'`, `'v'`, `'a'`, `'r'`), but our POS tags are Penn Treebank (NN, VBD, JJR, etc.). A simple mapping function translates by Penn tag's first letter: N→n, V→v, J→a, R→r, everything else falls back to 'n'.

**`lemma`** — WordNet lemma of `term_str`, computed using `wn_pos` as a hint. Lemma modifies only 32% of terms but handles morphological irregulars Porter cannot: `went → go`, `brought → bring`, `better → good`. Where Porter normalizes broadly but coarsely, WordNet normalizes selectively but linguistically.

### Key observation: Porter vs WordNet

The two normalization strategies represent different philosophical commitments. Porter is mechanical — strip recognizable suffixes by rule, regardless of whether the result is a real word. WordNet is dictionary-based — only modify a term if it's a known inflected form of another dictionary entry, then return the canonical form. The 64% vs 32% modification rate quantifies this difference. Inspecting cases where the two disagree (`went/went/go`, `better/better/good`, `studies/studi/study`) makes the tradeoff concrete and gives F4/F5 analyses a choice point: aggressive collapsing via Porter, or linguistically-precise collapsing via lemma.

### Output

- **VOCAB** (`data/F3/VOCAB.csv`): F2 columns plus `stop`, `p_stem`, `pos_max`, `wn_pos`, `lemma`.
- **TOKEN** (`data/F3/TOKEN.csv`): unchanged from F2, re-saved for stage continuity.

### Notes for downstream stages

The empty-string row at `term_id = 0` carries `pos_max = NaN` (no tokens with that term_str were tagged with a POS by NLTK — they were all punctuation that got stripped). It will continue to be filtered at analysis time. Function words occasionally get spurious lemmas (e.g., `as → a`) because WordNet has poor coverage of non-content words and `wn_pos` defaults them to 'n' — this is harmless because they're filtered as stopwords downstream. F4 will use VOCAB's `pos_max` for term-level analysis, `stop` for filtering, and either `p_stem` or `lemma` (analyst's choice) as the normalization key for TFIDF aggregation.

## Setup

In [27]:
import pandas as pd
import numpy as np
import os
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

F2_path = 'data/F2'
F3_path = 'data/F3'
os.makedirs(F3_path, exist_ok=True)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

OHCO = ['work_id','part_num','chap_num','para_num','sent_num','token_num']


## Load F2 Data

In [28]:
LIB = pd.read_csv(f"{F2_path}/../F1/LIB.csv", index_col=OHCO[0])
TOKEN = pd.read_csv(f"{F2_path}/TOKEN.csv", index_col=OHCO, keep_default_na = False, na_values = [])
VOCAB = pd.read_csv(f"{F2_path}/VOCAB.csv", index_col='term_id', keep_default_na = False, na_values = [])

print(f"LIB: {len(LIB)} works")
print(f"TOKEN: {len(TOKEN):,} tokens")
print(f"VOCAB: {len(VOCAB):,} terms")

LIB: 49 works
TOKEN: 283,995 tokens
VOCAB: 15,464 terms


In [29]:
VOCAB.head()

,term_str,n,num
term_id,,,
0,,115,0
1,0s,1,1
2,1,2,1
3,10,3,1
4,100000,1,1


## Add stopwords

In [30]:
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns = ['term_str'])
sw = sw.reset_index().set_index('term_str')
sw.columns = ['dummy']
sw.dummy = 1

VOCAB['stop'] = VOCAB.term_str.map(sw.dummy)
VOCAB['stop'] = VOCAB['stop'].fillna(0).astype('int')

print(f"Stopwords matched: {VOCAB['stop'].sum()} terms")
VOCAB[VOCAB['stop'] == 1].sample(10)

Stopwords matched: 131 terms


,term_str,n,num,stop
term_id,,,,
5284,few,180,0,1
9268,now,383,0,1
8875,more,537,0,1
9556,ours,37,0,1
1957,by,2113,0,1
6854,i,5498,0,1
11891,same,349,0,1
6601,himself,195,0,1
1356,because,298,0,1


## Add Porter Stems

In [31]:
stemmer = PorterStemmer()
VOCAB['p_stem'] = VOCAB.term_str.apply(stemmer.stem)

VOCAB.sample(10)

,term_str,n,num,stop,p_stem
term_id,,,,,
15015,weave,3,0,0,weav
4856,europeans,4,0,0,european
6908,illuminated,1,0,0,illumin
7271,insatiable,3,0,0,insati
13415,supplement,1,0,0,supplement
2986,contradicente,2,0,0,contradicent
6696,hoofs,3,0,0,hoof
5793,fully,22,0,0,fulli
3486,dearly,2,0,0,dearli


## Add pos_max (mos common POS per term)

In [32]:
pos_max = (TOKEN.groupby(['term_str','pos']).size()
           .unstack(fill_value=0)
           .idxmax(axis=1)
           .rename('pos_max'))

VOCAB = VOCAB.merge(pos_max, left_on = 'term_str',right_index = True, how = 'left')

VOCAB.pos_max.value_counts().head(15)

pos_max
NN     5833
JJ     2162
NNP    1645
NNS    1464
VB      925
VBG     737
VBN     706
VBD     556
RB      442
VBZ     308
VBP     194
CD      116
JJS     109
IN      106
JJR      47
Name: count, dtype: int64

## Add WordNet lemma

In [33]:
def penn_to_wordnet(tag):
    if not isinstance(tag, str):
        return 'n'
    first = tag[0]
    if first == 'V': return 'v'
    if first == 'J': return 'a'
    if first == 'R': return 'r'
    return 'n'

VOCAB['wn_pos'] = VOCAB.pos_max.apply(penn_to_wordnet)

lemmatizer = WordNetLemmatizer()
VOCAB['lemma'] = VOCAB.apply(lambda r: lemmatizer.lemmatize(r.term_str, r.wn_pos), axis = 1)

VOCAB.sample(10)

,term_str,n,num,stop,p_stem,pos_max,wn_pos,lemma
term_id,,,,,,,,
11805,rules,25,0,0,rule,NNS,n,rule
14126,travellingcloset,2,0,0,travellingcloset,NN,n,travellingcloset
3963,discern,4,0,0,discern,VB,v,discern
9454,oneandthirty,1,0,0,oneandthirti,JJ,a,oneandthirty
1011,assay,5,0,0,assay,NN,n,assay
13662,temperature,2,0,0,temperatur,NN,n,temperature
14204,trouble,39,0,0,troubl,NN,n,trouble
6920,imagination,22,0,0,imagin,NN,n,imagination
994,asiatic,1,0,0,asiat,NNP,n,asiatic


In [34]:
diff_lemma = (VOCAB.term_str != VOCAB.lemma).sum()
diff_stem = (VOCAB.term_str != VOCAB.p_stem).sum()
print(f"Terms whose lemma differs from term_str: {diff_lemma:,} ({diff_lemma/len(VOCAB):.1%})")
print(f"Terms whose stem  differs from term_str: {diff_stem:,} ({diff_stem/len(VOCAB):.1%})")


Terms whose lemma differs from term_str: 4,883 (31.6%)
Terms whose stem  differs from term_str: 9,946 (64.3%)


In [35]:
VOCAB[(VOCAB.p_stem != VOCAB.lemma) & (VOCAB.n > 50)].sample(15)[
    ['term_str','n','pos_max','p_stem','lemma']
]

,term_str,n,pos_max,p_stem,lemma
term_id,,,,,
5151,family,76,NN,famili,family
15161,wholly,109,RB,wholli,wholly
14606,usually,64,RB,usual,usually
2688,company,106,NN,compani,company
5248,feet,129,NNS,feet,foot
1493,better,163,JJR,better,good
13812,this,1670,DT,thi,this
7747,knowledge,63,NN,knowledg,knowledge
8239,lost,70,VBN,lost,lose
